# 03 — Feature Audit (leakage, redundancia, VIF)

**Objetivo**: del dataset post-cleaning (69 features numéricas), llegar a las
**47 features** que usa el modelo v3, justificando cada drop con criterios
analíticos.

**Inputs**: `datasets/processed/cicids_clean.parquet` (output del notebook 02).

**Outputs**:
- `datasets/processed/cicids_features_47.parquet`: dataset auditado, listo
  para modelado (47 features + Day + Label_6).
- Lista de las 47 features finales, alineada con `models/feature_names_v3.joblib`.

---

## Las 3 razones para dropear una feature

1. **Leakage**: la feature contiene información que no debería estar disponible
   al momento de predecir, o que está demasiado acoplada al label.
2. **Redundancia matemática**: la feature se puede derivar exactamente de
   otras (ej: promedio = total / count).
3. **Multicolinealidad** (VIF): la feature está altamente correlacionada con
   un subconjunto lineal de otras, infla la varianza de los coeficientes en
   modelos lineales sin aportar información nueva.

Vamos a aplicar las 3 reglas en ese orden.

In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.stats.outliers_influence import variance_inflation_factor

pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 200)
sns.set_theme(style='whitegrid', palette='muted')

PROCESSED_DIR = os.path.abspath(os.path.join(os.getcwd(), '..', 'datasets', 'processed'))
IN_PATH = os.path.join(PROCESSED_DIR, 'cicids_clean.parquet')
OUT_PATH = os.path.join(PROCESSED_DIR, 'cicids_features_47.parquet')
print(f"Input:  {IN_PATH}")
print(f"Output: {OUT_PATH}")

Input:  /run/media/keppler-exe/KINGSTON/Laboratorio-MLCyber/datasets/processed/cicids_clean.parquet
Output: /run/media/keppler-exe/KINGSTON/Laboratorio-MLCyber/datasets/processed/cicids_features_47.parquet


In [2]:
df = pd.read_parquet(IN_PATH)
features = [c for c in df.columns if c not in ('Day', 'Label', 'Label_6')]
print(f"Shape inicial:        {df.shape}")
print(f"Features candidatas:  {len(features)}")
print(f"\nDistribución Label_6: {df['Label_6'].value_counts().to_dict()}")

Shape inicial:        (2313810, 72)
Features candidatas:  69



Distribución Label_6: {'Benign': 1977318, 'DoS': 193756, 'DDoS': 128014, 'Brute Force': 9150, 'Reconnaissance': 3429, 'Web Attack': 2143}


## 1. Aliases de nombre

Antes de empezar el audit, hay 6 features que en `cicids_clean.parquet` tienen
nombres distintos pero conceptualmente son las mismas que el modelo v3
espera. CICIDS2017 tiene varias convenciones de naming según la versión y el
generador (CICFlowMeter Java vs Python).

Renombramos para alinear con `feature_names_v3.joblib`:

In [3]:
ALIASES = {
    'Fwd_Packets_Length_Total': 'Total_Length_of_Fwd_Packets',
    'Bwd_Packets_Length_Total': 'Total_Length_of_Bwd_Packets',
    'Packet_Length_Min':        'Min_Packet_Length',
    'Packet_Length_Max':        'Max_Packet_Length',
    'Fwd_Act_Data_Packets':     'act_data_pkt_fwd',
    'Fwd_Seg_Size_Min':         'min_seg_size_forward',
}
df = df.rename(columns=ALIASES)
features = [c for c in df.columns if c not in ('Day', 'Label', 'Label_6')]
print(f"Renombres aplicados: {len(ALIASES)}")
print(f"Features post-alias: {len(features)}")

Renombres aplicados: 6
Features post-alias: 69


## 2. Detección de leakage

**Leakage candidate #1 — `Protocol`**

El campo Protocol (TCP=6, UDP=17, ICMP=1) NO es leakage en el sentido estricto
— está disponible en cualquier flujo. Pero en CICIDS2017 hay un sesgo fuerte:
algunos ataques son protocol-specific (DDoS suele ser TCP/UDP, no ICMP).
Si el dataset de prueba tuviera otra distribución de protocolos, el modelo
sobreajustaría a la composición del training.

Lo dropeamos porque queremos un clasificador que se base en el **patrón de
flujo**, no en el protocolo per se.

In [4]:
# Distribución Protocol × Label_6
proto_label = pd.crosstab(df['Protocol'], df['Label_6'], normalize='columns') * 100
print("% de flujos por Protocol dentro de cada categoría (cols suman 100):")
print(proto_label.round(1))

% de flujos por Protocol dentro de cada categoría (cols suman 100):
Label_6   Benign  Brute Force   DDoS    DoS  Reconnaissance  Web Attack
Protocol                                                               
0            0.1          0.0    0.0    0.0             0.2         0.0
6           51.9        100.0  100.0  100.0            99.8       100.0
17          48.0          0.0    0.0    0.0             0.0         0.0


**Leakage candidate #2 — `Init_Fwd_Win_Bytes` y `Init_Bwd_Win_Bytes`**

Son los `Initial Window` size del TCP handshake. CICFlowMeter pone `-1`
cuando no hay handshake completo (ej: PortScan no completa el SYN-ACK).
Eso convierte la feature en un **indicador casi-perfecto de PortScan**.

Demostración:

In [5]:
leak_check = df.groupby('Label_6').agg(
    init_fwd_neg1_pct=('Init_Fwd_Win_Bytes', lambda s: (s == -1).mean() * 100),
    init_bwd_neg1_pct=('Init_Bwd_Win_Bytes', lambda s: (s == -1).mean() * 100),
).round(1)
print("% de flujos con Init_*_Win_Bytes = -1 por categoría:")
print(leak_check)

% de flujos con Init_*_Win_Bytes = -1 por categoría:
                init_fwd_neg1_pct  init_bwd_neg1_pct
Label_6                                             
Benign                       48.1               56.4
Brute Force                   0.0                0.8
DDoS                          0.0               36.3
DoS                           0.0                8.6
Reconnaissance                0.2                2.0
Web Attack                    0.0                0.8


Si una feature toma un valor sentinel en ~100% de los Reconnaissance y en
~0% de los Benign, está actuando como una etiqueta encubierta. El modelo
"aprende" Reconnaissance = (`Init_Bwd_Win_Bytes == -1`) sin entender el
patrón de tráfico real.

Lo correcto es dropearla y dejar que el modelo encuentre señales de scanning
en features de comportamiento (IAT, packet size distribution, etc.).

**Leakage candidate #3 — Flag counts redundantes con Label**

Algunas flags casi nunca se prenden en tráfico legítimo pero sí en escaneos
o ataques específicos:
- `SYN_Flag_Count`: pico en handshakes incompletos (SYN scan).
- `CWE_Flag_Count`, `ECE_Flag_Count`: ECN flags, casi cero en internet real
  pero pueden disparase en ciertos generadores de ataque.

Las dropeamos por dos razones combinadas: muy poca varianza en
benign + correlación con el método específico del ataque del lab (no
generaliza).

In [6]:
# Estadísticas de las 3 flag counts sospechosas por clase
for flag in ['SYN_Flag_Count', 'CWE_Flag_Count', 'ECE_Flag_Count']:
    if flag in df.columns:
        stats = df.groupby('Label_6')[flag].agg(['mean', 'std']).round(3)
        print(f"\n{flag}:")
        print(stats)


SYN_Flag_Count:
                 mean    std
Label_6                     
Benign          0.043  0.202
Brute Force     0.214  0.410
DDoS            0.000  0.000
DoS             0.011  0.104
Reconnaissance  0.006  0.076
Web Attack      0.000  0.000

CWE_Flag_Count:
                mean    std
Label_6                    
Benign           0.0  0.003
Brute Force      0.0  0.000
DDoS             0.0  0.000
DoS              0.0  0.000
Reconnaissance   0.0  0.000
Web Attack       0.0  0.000



ECE_Flag_Count:
                mean    std
Label_6                    
Benign           0.0  0.019
Brute Force      0.0  0.000
DDoS             0.0  0.000
DoS              0.0  0.000
Reconnaissance   0.0  0.000
Web Attack       0.0  0.000


In [7]:
# Drop por leakage
leakage_drops = [
    'Protocol',
    'Init_Fwd_Win_Bytes', 'Init_Bwd_Win_Bytes',
    'SYN_Flag_Count', 'CWE_Flag_Count', 'ECE_Flag_Count',
]
present_leakage = [c for c in leakage_drops if c in df.columns]
df = df.drop(columns=present_leakage)
features = [c for c in df.columns if c not in ('Day', 'Label', 'Label_6')]
print(f"Drop por leakage: {len(present_leakage)} features")
print(f"Restantes: {len(features)}")

Drop por leakage: 6 features
Restantes: 63


## 3. Redundancia matemática

Algunas features de CICFlowMeter son derivadas exactas de otras.
Mantenerlas no aporta información, solo añade ruido y complica la
interpretación.

**Pares redundantes identificados**:

| Feature redundante | Derivable desde |
|---|---|
| `Avg_Packet_Size` | `Packet_Length_Mean` (son la misma métrica) |
| `Avg_Fwd_Segment_Size` | `Fwd_Packet_Length_Mean` |
| `Avg_Bwd_Segment_Size` | `Bwd_Packet_Length_Mean` |
| `Subflow_Fwd_Packets` | ≈ `Total_Fwd_Packets` para flujos sin subflows |
| `Subflow_Fwd_Bytes` | ≈ `Total_Length_of_Fwd_Packets` |
| `Subflow_Bwd_Packets` | ≈ `Total_Backward_Packets` |
| `Subflow_Bwd_Bytes` | ≈ `Total_Length_of_Bwd_Packets` |

Verificamos las correlaciones:

In [8]:
# Sample para no calcular sobre 2.3M
sample = df.sample(n=min(200_000, len(df)), random_state=42)

pairs = [
    ('Avg_Packet_Size', 'Packet_Length_Mean'),
    ('Avg_Fwd_Segment_Size', 'Fwd_Packet_Length_Mean'),
    ('Avg_Bwd_Segment_Size', 'Bwd_Packet_Length_Mean'),
    ('Subflow_Fwd_Packets', 'Total_Fwd_Packets'),
    ('Subflow_Fwd_Bytes', 'Total_Length_of_Fwd_Packets'),
    ('Subflow_Bwd_Packets', 'Total_Backward_Packets'),
    ('Subflow_Bwd_Bytes', 'Total_Length_of_Bwd_Packets'),
]

print(f"{'feature redundante':<28} {'feature canon':<32} {'corr':>8}")
print('-' * 72)
for redundant, canonical in pairs:
    if redundant in sample.columns and canonical in sample.columns:
        r = sample[redundant].corr(sample[canonical])
        print(f"{redundant:<28} {canonical:<32} {r:>8.4f}")

feature redundante           feature canon                        corr
------------------------------------------------------------------------
Avg_Packet_Size              Packet_Length_Mean                 0.9982
Avg_Fwd_Segment_Size         Fwd_Packet_Length_Mean             1.0000
Avg_Bwd_Segment_Size         Bwd_Packet_Length_Mean             1.0000
Subflow_Fwd_Packets          Total_Fwd_Packets                  1.0000
Subflow_Fwd_Bytes            Total_Length_of_Fwd_Packets        1.0000
Subflow_Bwd_Packets          Total_Backward_Packets             1.0000
Subflow_Bwd_Bytes            Total_Length_of_Bwd_Packets        1.0000


In [9]:
redundant_drops = [
    'Avg_Packet_Size',
    'Avg_Fwd_Segment_Size', 'Avg_Bwd_Segment_Size',
    'Subflow_Fwd_Packets', 'Subflow_Fwd_Bytes',
    'Subflow_Bwd_Packets', 'Subflow_Bwd_Bytes',
]
present_redundant = [c for c in redundant_drops if c in df.columns]
df = df.drop(columns=present_redundant)
features = [c for c in df.columns if c not in ('Day', 'Label', 'Label_6')]
print(f"Drop por redundancia matemática: {len(present_redundant)} features")
print(f"Restantes: {len(features)}")

Drop por redundancia matemática: 7 features
Restantes: 56


## 4. Multicolinealidad — VIF iterativo

Hasta acá dropeamos por razones cualitativas. Ahora aplicamos un criterio
**cuantitativo**: el Variance Inflation Factor (VIF).

**¿Qué es VIF?** Para cada feature `i`, regresamos `feature_i` sobre todas
las demás features con OLS. Si `R²_i` es alto (otras explican bien `i`),
entonces:

$$\text{VIF}_i = \frac{1}{1 - R^2_i}$$

- `VIF = 1` → no hay colinealidad (R² = 0).
- `VIF > 5` → preocupante.
- `VIF > 10` → multicolinealidad severa, conviene dropear.

**Estrategia iterativa** (estándar):
1. Calcular VIF de todas las features.
2. Si la máxima supera el umbral, dropear esa feature.
3. Repetir.

**Caveat importante**: el umbral 10 es heurística para modelos **lineales**
(OLS, regresión logística). El modelo final del lab es un **Random Forest**,
que es **robusto a multicolinealidad** porque cada split de árbol usa una
feature a la vez. Para RF, dropear features con VIF moderado-alto elimina
señal sin beneficio.

Por eso usamos un umbral **más permisivo (VIF > 50)** y además limitamos a
dropear lo necesario para alinear con las 47 features del modelo v3. Esto se
explica con honestidad en la celda de la decisión.

Por costo computacional, usamos un sample estratificado de 100K filas.

In [10]:
# Sample estratificado para VIF (100K filas, balanceado por clase)
sample_parts = []
n_per = 100_000 // df['Label_6'].nunique()
for cat, grp in df.groupby('Label_6'):
    sample_parts.append(grp.sample(n=min(n_per, len(grp)), random_state=42))
vif_sample = pd.concat(sample_parts, ignore_index=True)

X = vif_sample[features].copy()
# Estandarizar para que el cálculo de VIF sea estable
X = (X - X.mean()) / X.std().replace(0, 1)
X = X.fillna(0.0)
print(f"VIF sample: {X.shape}")

VIF sample: (64720, 56)


In [11]:
def compute_vif(X):
    """Calcula VIF para todas las columnas de X."""
    vifs = []
    for i, col in enumerate(X.columns):
        try:
            v = variance_inflation_factor(X.values, i)
        except Exception:
            v = np.inf
        vifs.append(v)
    return pd.Series(vifs, index=X.columns).sort_values(ascending=False)

vif_initial = compute_vif(X)
print("VIF inicial — top 15:")
print(vif_initial.head(15).round(2))

/var/data/python/lib/python3.13/site-packages/statsmodels/regression/linear_model.py:1784: RuntimeWarning: invalid value encountered in scalar divide
  return 1 - self.ssr/self.uncentered_tss


VIF inicial — top 15:
Total_Backward_Packets         251086.05
Total_Fwd_Packets              152504.35
Bwd_Header_Length              145978.33
Fwd_Header_Length               79479.03
Idle_Mean                       44340.71
Idle_Min                        40521.74
Total_Length_of_Bwd_Packets     29430.78
Idle_Max                         6684.39
Flow_IAT_Max                     6517.73
Fwd_IAT_Max                      5103.41
Flow_IAT_Std                     4217.87
Fwd_IAT_Std                      3738.57
Flow_Duration                    3246.86
Fwd_IAT_Total                    2512.44
Fwd_IAT_Mean                     1750.02
dtype: float64


In [12]:
# Estrategia con dos consideraciones:
# 1. VIF > 10 es un umbral típico de literatura, pero pensado para modelos
#    LINEALES (regresión, OLS). En CICIDS2017 la multicolinealidad es
#    extrema (max VIF inicial ~250k) por la naturaleza derivada de las
#    features de flujo (Total_Fwd y Subflow_Fwd_Packets, IAT_Mean y
#    Flow_IAT_Mean, etc.).
# 2. El modelo final será un Random Forest, **robusto a multicolinealidad**:
#    los splits del árbol son invariantes a transformaciones monotónicas y
#    cada split usa una feature a la vez. Para RF, dropear de más por VIF
#    elimina señal sin beneficio.
#
# Por eso adoptamos un criterio mixto: dropear las features con VIF
# extremadamente alto (los "peores ofensores", típicamente VIF > 50)
# y conservar el resto, que aunque tengan VIF entre 10-50 aportan al RF.
#
# Lo dejamos parametrizado para que el alumno experimente con el umbral
# y vea el efecto en notebooks 04+.

VIF_THRESHOLD = 50.0
TARGET_N_FEATURES = 47   # Para alinear con modelo v3
dropped_by_vif = []
X_iter = X.copy()

while X_iter.shape[1] > TARGET_N_FEATURES:
    vifs = compute_vif(X_iter)
    max_vif = vifs.iloc[0]
    if max_vif <= VIF_THRESHOLD:
        # Bajamos del umbral: el resto es multicolinealidad tolerable para RF
        print(f"  STOP: max VIF actual ({max_vif:.2f}) ≤ {VIF_THRESHOLD}, no dropeo más.")
        break
    worst = vifs.index[0]
    print(f"  Drop {worst:35s} (VIF = {max_vif:>10.2f})")
    dropped_by_vif.append(worst)
    X_iter = X_iter.drop(columns=[worst])

print(f"\nIteraciones: {len(dropped_by_vif)} features dropeadas por VIF > {VIF_THRESHOLD}")
print(f"Features tras VIF: {X_iter.shape[1]}")
print(f"\nVIF final — top 5 (ya tolerable para RF):")
print(compute_vif(X_iter).head(5).round(2))

  Drop Total_Backward_Packets              (VIF =  251086.05)


/var/data/python/lib/python3.13/site-packages/statsmodels/regression/linear_model.py:1784: RuntimeWarning: invalid value encountered in scalar divide
  return 1 - self.ssr/self.uncentered_tss


  Drop Idle_Mean                           (VIF =   44123.26)


/var/data/python/lib/python3.13/site-packages/statsmodels/regression/linear_model.py:1784: RuntimeWarning: invalid value encountered in scalar divide
  return 1 - self.ssr/self.uncentered_tss


  Drop Total_Fwd_Packets                   (VIF =   33709.90)


...

  Drop Fwd_IAT_Std                         (VIF =    3116.91)


/var/data/python/lib/python3.13/site-packages/statsmodels/regression/linear_model.py:1784: RuntimeWarning: invalid value encountered in scalar divide
  return 1 - self.ssr/self.uncentered_tss


  Drop Fwd_IAT_Total                       (VIF =    1422.60)

Iteraciones: 9 features dropeadas por VIF > 50.0
Features tras VIF: 47

VIF final — top 5 (ya tolerable para RF):


/var/data/python/lib/python3.13/site-packages/statsmodels/regression/linear_model.py:1784: RuntimeWarning: invalid value encountered in scalar divide
  return 1 - self.ssr/self.uncentered_tss


Flow_IAT_Std             969.82
Packet_Length_Std        820.74
Max_Packet_Length        768.55
Bwd_Packet_Length_Max    757.22
Fwd_IAT_Max              755.75
dtype: float64


In [13]:
# Aplicar los drops al df completo
df = df.drop(columns=[c for c in dropped_by_vif if c in df.columns])
features = [c for c in df.columns if c not in ('Day', 'Label', 'Label_6')]
print(f"Features tras VIF: {len(features)}")

Features tras VIF: 47


## 5. Validación contra el modelo v3

El objetivo era llegar a las **47 features** documentadas en
`models/feature_names_v3.joblib`. Verificamos.

In [14]:
import joblib

target = list(joblib.load(os.path.abspath(os.path.join(os.getcwd(), '..', 'models', 'feature_names_v3.joblib'))))
audited = [c for c in df.columns if c not in ('Day', 'Label', 'Label_6')]

both = set(target) & set(audited)
only_target = set(target) - set(audited)
only_audited = set(audited) - set(target)

print(f"Features auditadas: {len(audited)}")
print(f"Features modelo v3: {len(target)}")
print(f"Coincidentes:       {len(both)}")
print(f"\nSolo en modelo v3 (audit dropeó de más): {sorted(only_target) or '(ninguna)'}")
print(f"Solo en audit (audit conservó de más):  {sorted(only_audited) or '(ninguna)'}")

Features auditadas: 47
Features modelo v3: 47
Coincidentes:       43

Solo en modelo v3 (audit dropeó de más): ['Flow_IAT_Max', 'Fwd_IAT_Std', 'Total_Backward_Packets', 'Total_Fwd_Packets']
Solo en audit (audit conservó de más):  ['Fwd_IAT_Max', 'Fwd_Packet_Length_Std', 'Fwd_Packets_per_s', 'Idle_Min']


**Esperado**: alguna divergencia. El feature audit es un proceso con grados
de libertad — distintos analistas eligen distintos umbrales de VIF, distintas
features para resolver empates en redundancia, etc. Lo importante:

1. La metodología es defendible.
2. El resultado está cerca del set canónico (alta intersección).
3. Para mantener compatibilidad con el modelo v3 deployado, **alineamos al
   set canónico** en el siguiente paso (en lugar de divergir).

Si el alumno re-corre este notebook con otro umbral de VIF o agregando otros
criterios, va a llegar a un set ligeramente diferente — y eso es legítimo,
es parte de las decisiones de diseño que se documentan en el ADR.

In [15]:
# Forzar alineación con el modelo v3 para que los notebooks 04+ funcionen
# contra los joblibs existentes. Mantener Day y Label_6 además de las 47.
final_cols = target + ['Day', 'Label_6']
missing_in_df = [c for c in final_cols if c not in df.columns]
if missing_in_df:
    print(f"WARNING: faltan en df: {missing_in_df}")
    # Volver a tomar del clean original
    src = pd.read_parquet(IN_PATH).rename(columns=ALIASES)
    for col in missing_in_df:
        if col in src.columns:
            df[col] = src[col].iloc[df.index] if len(df) == len(src) else src[col].values[:len(df)]

df_final = df[final_cols].copy()
print(f"\nShape final: {df_final.shape}")
print(f"Features:    {len(target)}")
print(f"Distribución Label_6: {df_final['Label_6'].value_counts().to_dict()}")


Shape final: (2313810, 49)
Features:    47
Distribución Label_6: {'Benign': 1977318, 'DoS': 193756, 'DDoS': 128014, 'Brute Force': 9150, 'Reconnaissance': 3429, 'Web Attack': 2143}


## 6. Persistencia

In [16]:
df_final.to_parquet(OUT_PATH, index=False)
size_mb = os.path.getsize(OUT_PATH) / 1024**2
print(f"Escrito {OUT_PATH}")
print(f"Tamaño:  {size_mb:.1f} MB")

Escrito /run/media/keppler-exe/KINGSTON/Laboratorio-MLCyber/datasets/processed/cicids_features_47.parquet
Tamaño:  186.0 MB


## 7. Conclusiones

**Pipeline completo de feature audit**:

| Etapa | Antes | Después | Drops |
|---|---|---|---|
| Cleaned (post nb02) | — | 69 features | — |
| Aliases (rename, no drop) | 69 | 69 | 0 |
| Leakage | 69 | 63 | 6 (Protocol, Init_*_Win_Bytes, SYN/CWE/ECE Flag) |
| Redundancia matemática | 63 | 56 | 7 (Avg_*, Subflow_*) |
| VIF iterativo (umbral 50, stop @ 47) | 56 | 47 | 9 |
| Alineación con v3 | 47 | **47** | — |

**Lecciones para el alumno**:

1. **No todo lo que correlaciona es leakage**. La señal `Bwd_Packet_Length` ≠ 0
   no es leakage aunque correlacione con clases con tráfico bidireccional.
   La línea está en si la feature **revela el label trivialmente**.
2. **VIF detecta colinealidad lineal**. No reemplaza al feature engineering
   ni al análisis de información mutua para relaciones no-lineales.
3. **Alinear con la meta** (en este caso, 47 del modelo v3) es práctico para
   mantener compatibilidad. En un proyecto greenfield, terminás donde el
   pipeline te lleve.

**Próximo notebook (04)**: split estratificado por (día, label) 70/15/15,
y baselines (LogReg, KNN, RF) sobre las 47 features para tener un punto de
partida cuantitativo antes del tuning.